# Prompt 04: Chain-of-Thought & Reasoning

1. Direct vs zero-shot CoT on a math task
2. Few-shot CoT with worked examples
3. Self-consistency (N samples, majority vote)
4. Hidden CoT with `<thinking>` tags
5. Exercises: Tree-of-Thoughts sketch; measure CoT lift on your tasks

Deps: `pip install anthropic openai`

## 1. Setup

In [ ]:
import os, re

def call(prompt, system='You solve problems step by step.', temperature=0, max_tokens=600):
    if os.getenv('ANTHROPIC_API_KEY'):
        from anthropic import Anthropic
        r = Anthropic().messages.create(model='claude-sonnet-4-6', max_tokens=max_tokens,
            temperature=temperature, system=system,
            messages=[{'role':'user','content':prompt}])
        return r.content[0].text
    if os.getenv('OPENAI_API_KEY'):
        from openai import OpenAI
        r = OpenAI().chat.completions.create(model='gpt-4o-mini', max_tokens=max_tokens, temperature=temperature,
            messages=[{'role':'system','content':system},{'role':'user','content':prompt}])
        return r.choices[0].message.content
    return '[no provider]'

def extract_final_number(text):
    # Grab the last number in the output (often the final answer)
    nums = re.findall(r'-?\d+(?:\.\d+)?', text.replace(',',''))
    return float(nums[-1]) if nums else None

## 2. Tasks to Benchmark On

Small GSM8K-style set.

In [ ]:
problems = [
    ('A school sells tickets at $8 each. They sell 127 tickets. Later they refund 12. Revenue?', 8 * (127 - 12)),
    ('A recipe needs 3/4 cup sugar. If you triple the recipe, how much sugar?', 3 * 3/4),
    ('An engineer earns $140,000/yr. Raises of 6% each year for 3 years. New salary?',
     round(140000 * (1.06**3), 2)),
    ('A car travels 60 mph for 2.5 hours, then 45 mph for 1.25 hours. Total miles?',
     60*2.5 + 45*1.25),
    ('There are 12 boxes, each with 18 cookies. 23 are eaten. How many remain?', 12*18 - 23),
    ('Order total is $45. 8% sales tax + 18% tip on subtotal. Grand total?',
     round(45 * (1 + 0.08 + 0.18), 2)),
]

## 3. Direct vs Zero-Shot CoT

In [ ]:
direct_hits = cot_hits = 0
for q, gold in problems:
    direct = call(q + '\n\nAnswer with just the number.', temperature=0)
    cot = call(q + '\n\nLet\'s think step by step. At the end write FINAL: <number>.', temperature=0)
    d = extract_final_number(direct); c = extract_final_number(cot)
    ok_d = d is not None and abs(d - gold) < 0.05
    ok_c = c is not None and abs(c - gold) < 0.05
    direct_hits += int(ok_d); cot_hits += int(ok_c)
    print(f'{"✓" if ok_d else "✗"} direct={d}   {"✓" if ok_c else "✗"} CoT={c}   gold={gold}')

print(f'\ndirect: {direct_hits}/{len(problems)}   CoT: {cot_hits}/{len(problems)}')

## 4. Few-Shot CoT

In [ ]:
few_shot_preamble = '''Examples:

Q: A shop sells pens at $2 each. They sell 40 and refund 5. Revenue?
A: 40 - 5 = 35 net sales. 35 * $2 = $70. FINAL: 70

Q: A room is 12 by 15 feet. What is its area?
A: Area = length * width = 12 * 15 = 180 sq ft. FINAL: 180

Now solve:
Q: {question}
A:'''

fs_hits = 0
for q, gold in problems:
    fs = call(few_shot_preamble.format(question=q), temperature=0)
    v = extract_final_number(fs)
    ok = v is not None and abs(v - gold) < 0.05
    fs_hits += int(ok)
    print(f'{"✓" if ok else "✗"} fs-CoT={v}   gold={gold}')
print(f'\nfew-shot CoT: {fs_hits}/{len(problems)}')

## 5. Self-Consistency

In [ ]:
from collections import Counter

def self_consistency(question, n_samples=5):
    votes = []
    for _ in range(n_samples):
        out = call(question + '\n\nLet\'s think step by step. At the end write FINAL: <number>.',
                   temperature=0.7)
        v = extract_final_number(out)
        if v is not None:
            votes.append(round(v, 2))
    return Counter(votes).most_common(1)[0][0] if votes else None

sc_hits = 0
for q, gold in problems:
    v = self_consistency(q, n_samples=5)
    ok = v is not None and abs(v - gold) < 0.05
    sc_hits += int(ok)
    print(f'{"✓" if ok else "✗"} sc={v}   gold={gold}')
print(f'\nself-consistency (n=5): {sc_hits}/{len(problems)}')

## 6. Hidden CoT: Separate Thinking from Answer

In [ ]:
tagged = '''Solve this. Put your reasoning inside <thinking>...</thinking>, then your final
answer inside <answer>...</answer>. Show only the number inside <answer>.

Q: {q}'''

q = 'A bakery sells 120 croissants at $3.50 each. They discount 25 to $2 each due to day-old. Revenue?'
raw = call(tagged.format(q=q), temperature=0)
print('--- raw ---\n', raw)

thinking = re.search(r'<thinking>(.*?)</thinking>', raw, re.S)
answer  = re.search(r'<answer>(.*?)</answer>', raw, re.S)
print('\n--- extracted ---')
print('thinking (for logs):', thinking.group(1).strip()[:200] if thinking else None)
print('answer   (for user):', answer.group(1).strip() if answer else None)

## 7. Exercise: Tree-of-Thoughts Sketch

For a puzzle task (e.g., Game of 24, cryptarithm), implement:
```python
def tot_solve(problem, depth=3, branch=3):
    frontier = [initial_state(problem)]
    for _ in range(depth):
        next_frontier = []
        for state in frontier:
            children = generate_children(state, k=branch)   # LLM call: propose moves
            scored = [(eval_state(c), c) for c in children]   # LLM call: evaluate
            next_frontier += [c for _, c in sorted(scored, reverse=True)[:branch]]
        frontier = next_frontier
    return best_solution(frontier)
```

Measure vs plain CoT on a 20-problem set. You'll see ToT winning on hard search tasks but paying ≥10x tokens.

## 8. Exercise: Does CoT Help YOUR Tasks?

Pick 3 features from your product. For each:
1. 20-example eval set.
2. Compare direct, zero-shot CoT, few-shot CoT, self-consistency.
3. Log accuracy, avg tokens, avg latency.

Only ship the highest mode where marginal accuracy gains ≥ marginal cost pain.

## 9. Takeaways

- **CoT helps on reasoning tasks**, not all tasks.
- **Self-consistency is the best-ROI enhancement** when you can afford N samples.
- **Reasoning models** (o1, Claude thinking, R1) obviate manual CoT on hard tasks.
- **Hide the chain** in production; keep it in logs.
- **Verify** — plausible traces can wrap wrong answers.

Next: **05 — Role Prompting & Personas**.